# `ask_anything()` — the capability typology as one executable function

One field, any question. The system decides the bucket and answers **only in the way that
bucket can be answered faithfully**:

```
ask_anything(question)
 |- change-over-time form? yes -> BUCKET 4  diachronic wording change:
 |       keyness evidence + KWIC quotes + guarded summary (evidence-only vocabulary,
 |       deterministic word-guard, template fallback -- never free generation)
 |- router: aggregate form?  no  -> BUCKET 1  retrieval -> grounded answer + citations
 `- yes -> which column does the answer need?
     |- validated CONTENT column (alienation_alleged)  -> BUCKET 3 (extracted):
     |       threshold-aware count + abstention audit
     |- METADATA columns                                -> BUCKET 2: NL->SQL with guardrails
     |       (coder model, few-shot, temp 0, EXPLAIN-validated, SQL printed for review)
     `- no column carries the answer                    -> BUCKET 3 (unextracted): refusal
```

Nothing here is new machinery: the router, retrieval, DuckDB layer, threshold helper **and
the NL->SQL translator** are exec'd from `rag_echr_ris.ipynb` and `echr_query.ipynb`; the
Bucket-4 keyness design is `diachronic_analysis.ipynb` made parameterisable (any topic,
any interval) and guarded. The metadata branch is genuinely semantic — the local coder
model maps "against Poland" to `respondent_state = 'POL'` and "Kanton Bern" to the Swiss
table itself — behind **two deterministic refusal nets**: a deny-list of known-unextracted
content concepts (semantic knowledge no validator can derive) and `EXPLAIN` schema
validation of every generated statement. Refusal is never delegated to the model —
measured: a 3B model offered a CANNOT_ANSWER escape takes it whenever the question needs
composition. The generated SQL always prints **above** its result: for aggregate answers
the trust boundary is the query text, not the number. The citable quantitative path
remains the reviewed canned queries.

## 1. Load the deployed pipeline (RAG cells + query-layer cells, cache-hot)

In [6]:
import json
from pathlib import Path

RAG_NB   = Path("rag_echr_ris.ipynb")
QUERY_NB = Path("echr_query.ipynb")

def exec_cells(nb_path, markers):
    nb = json.loads(nb_path.read_text())
    n = 0
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        if any(m in src for m in markers):
            exec(src, globals())
            n += 1
    print(f"exec'd {n} cells from {nb_path.name}")

# RAG pipeline: config, loaders, chunker, genre, corpus, embedder/index, retrieve,
# router (Layer 1), generation + answer(), provenance/KWIC audit
exec_cells(RAG_NB, ["DATA_DIR  = Path", "def load_json_records", "ECHR_ANCHORS = [",
                    "reusable ECHR genre helper", "records, chunks = [], []",
                    "_embedder = None", "def _to_hit", "AGGREGATE_PATTERNS = [",
                    "SYSTEM_PROMPT = (", "def kwic_extracts"])
assert index is not None, "index missing -- run rag_echr_ris.ipynb once first"

# Query layer: config, table -> DuckDB, threshold-aware helper, NL->SQL translator
exec_cells(QUERY_NB, ["MIN_CONFIDENCE = 0.70", "def _load_table", "def alienation_at(",
                      "def nl2sql"])
assert con is not None, "DuckDB table missing -- run echr_extraction + theme classify first"

print(f"ready: {index.ntotal} chunks | {len(df)} table rows | router: {len(AGGREGATE_PATTERNS)} patterns")

inputs: ['echr_parental_alienation.json', 'ris_parental_alienation.json', 'swiss_parental_alienation.json']
chunk=600w/80o | ECHR skip={'OPINION', 'RELEVANT_LAW'} law_only=False | dev_cap=None
RIS decisions=True (civil only) | Swiss=True
genre routing: exclude={'communicated'} | MMR fetch_k=30 lambda=0.7
normalisers ready: ['echr', 'ris', 'swiss']
section splitter + chunker ready
genre helper ready: ('merits', 'admissibility', 'communicated', 'other') | low-info: {'communicated'}
  echr : 1116 records from echr_parental_alienation.json
  ris : 38 principles + 479 civil decisions (dropped 31 criminal-senate/AUSL 'Text' records — Entfremdung homonym / ECtHR summaries)
  ris  : 517 records from ris_parental_alienation.json
  swiss: dropped 24 Rechenschaftsbericht records (court annual reports, not case law — multi-case digests that flood the top-k)
  swiss: 2007 records from swiss_parental_alienation.json

ECHR dates: 1114/1116 have YYYY-MM-DD, 2 n.d.
ECHR sections: 873/1116 parsed; 243 f

## 2. Metadata relations for the Bucket-2 handlers
Quiet re-registration of the cross-jurisdiction metadata (same fields as `echr_query.ipynb`
section 6b; the coverage audit lives there).

In [ ]:
import json as _json
import re as _re
import pandas as pd

_swiss = _json.loads((DATA_DIR / "swiss_parental_alienation.json").read_text())
_ris   = _json.loads((DATA_DIR / "ris_parental_alienation.json").read_text())

swiss_meta = pd.DataFrame([{"id": r.get("stable_id"), "canton": r.get("canton"),
                            "court_type": r.get("court_type"), "year": r.get("year")}
                           for r in _swiss])

_SEN = _re.compile(r"\d{1,3}\s*([A-Z][a-z]{1,2})")

def _senate(gz):
    m = _SEN.match((gz or "").split(";")[0].strip())
    return m.group(1) if m else None

ris_meta = pd.DataFrame([{
    "id": r.get("id"), "dokumenttyp": r.get("dokumenttyp"),
    "year": int(r["entscheidungsdatum"][:4]) if (r.get("entscheidungsdatum") or "")[:4].isdigit() else None,
    "senate": _senate(r.get("geschaeftszahl")),
} for r in _ris])

# ECHR metadata incl. the 2026-07-07 fields (formation, introduction->judgment duration)
from datetime import datetime as _dt
_echr_raw = _json.loads((DATA_DIR / "echr_parental_alienation.json").read_text())

def _pdate(s):
    s = (s or "").split()[0] if s else ""
    try:
        return _dt.strptime(s, "%d/%m/%Y").date()
    except (ValueError, IndexError):
        return None

def _formation(dc):
    for f in ("GRANDCHAMBER", "CHAMBER", "COMMITTEE"):
        if f in (dc or ""):
            return f
    return None

def _dur(r):
    if (r.get("doctype") or "").upper() == "HECOM":
        return None                      # communicated = still pending, no duration
    a = _pdate(r.get("introductiondate"))
    b = _pdate(r.get("judgementdate")) or _pdate(r.get("decisiondate"))
    return (b - a).days if a and b and b >= a else None

# extraction-layer dates (echr table): HUDOC metadata leaves most HECOMs undated
# (36/238 have judgementdate); the extraction layer parsed the communication date
# from the document text (236/238). Reuse it as the last fallback, same convention.
_extr_year = {r.id: int(str(r.judgment_date)[:4]) for r in df.itertuples()
              if str(r.judgment_date)[:4].isdigit()}

def _year(r):
    """ECLI year when judged; else HUDOC date fields; else extraction-layer date.
    For HECOM (no ECLI until judged) the year is the COMMUNICATION year."""
    e = r.get("ecli") or ""
    if e.startswith("ECLI:CE:ECHR:") and e.split(":")[3][:4].isdigit():
        return int(e.split(":")[3][:4])
    d = _pdate(r.get("judgementdate")) or _pdate(r.get("decisiondate"))
    return d.year if d else _extr_year.get(r.get("itemid"))

echr_meta = pd.DataFrame([{
    "id": r.get("itemid"), "respondent": r.get("respondent"),
    "importance": int(r["importance"]) if str(r.get("importance", "")).isdigit() else None,
    "year": _year(r),
    "formation": _formation(r.get("documentcollectionid")),
    "duration_days": _dur(r),
    "separate_opinion": str(r.get("separateopinion", "")).upper() == "TRUE",
} for r in _echr_raw])
# Every other HUDOC field, as its OWN table rather than more columns on `echr_meta`.
# `echr_meta` is named in the NL->SQL schema, and Bucket 2's last safety net is that DuckDB
# rejects a column the model invented: widening the table the translator knows about would let
# a hallucinated `echr_meta.article` filter resolve quietly instead of failing EXPLAIN. So the
# raw layer lands here, joined by the scope view (cell 8) and named in no prompt.
# The only decisions taken are mechanical -- scalar values, short ones (the cap is what drops
# `full_text`), nothing restating a curated column. What is QUERYABLE among them is not decided
# here: the profiler reads it off the data, so identifiers and free text fall out on their own.
_HUDOC_VALUE_MAXLEN = 200           # characters; HUDOC's longest real metadata value is ~120


def _hudoc_column(key):
    """The raw values for one HUDOC key, or None if they are not short scalars."""
    vals = [r.get(key) for r in _echr_raw]
    if any(v is not None and not isinstance(v, (str, int, float, bool)) for v in vals):
        return None
    if max((len(str(v)) for v in vals if v is not None), default=0) > _HUDOC_VALUE_MAXLEN:
        return None
    if all(v is None or isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals):
        return vals
    return [None if v is None or str(v).strip() == "" else str(v) for v in vals]


echr_hudoc = pd.DataFrame({"id": [r.get("itemid") for r in _echr_raw]})
for _k in sorted({k for r in _echr_raw for k in r}):
    if _k != "itemid" and (_col := _hudoc_column(_k)) is not None:
        echr_hudoc[_k] = _col

echr_kp = pd.DataFrame([
    {"id": r.get("itemid"), "kp_code": code.strip()}
    for r in _echr_raw for code in (r.get("kpthesaurus") or "").split(";") if code.strip()
])

con.register("swiss_meta", swiss_meta)
con.register("ris_meta", ris_meta)
con.register("echr_meta", echr_meta)
con.register("echr_hudoc", echr_hudoc)
con.register("echr_kp", echr_kp)

# cross-source keyword hits: every case's matched import keywords, queryable + joinable.
# (all four sources were keyword-imported; this exposes that, per source, as a long table.)
def _kw_rows(recs, idkey, source, juris):
    out = []
    for r in recs:
        i = r.get(idkey)
        for k in (r.get("matched_keywords") or []):
            if i and k:
                out.append({"id": i, "source": source, "jurisdiction": juris, "keyword": k})
    return out
keyword_hits = pd.DataFrame(_kw_rows(_echr_raw, "itemid", "echr", "ECHR")
                            + _kw_rows(_swiss, "stable_id", "swiss", "CH")
                            + _kw_rows(_ris, "id", "ris", "AT"))
con.register("keyword_hits", keyword_hits)

# cross-source ZERO-SHOT themes (portable classifier theme_classify_zeroshot.py; approximate,
# not validated). One row per (case, source) with primary_theme + is_<theme> flags for ALL
# sources -> theme questions ("contact_access per canton") become answerable cross-jurisdiction.
try:
    _zt = pd.read_parquet(DATA_DIR / "themes_zeroshot.parquet")
    _zt_is = [c for c in _zt.columns if c.startswith("zs_is_")]
    _rename = {"primary_theme_zs": "primary_theme", "theme_group_zs": "theme_group"}
    _rename.update({c: c.replace("zs_is_", "is_") for c in _zt_is})
    case_themes = _zt.rename(columns=_rename)[
        ["id", "source", "jurisdiction", "primary_theme", "theme_group"]
        + [c.replace("zs_is_", "is_") for c in _zt_is]]
    con.register("case_themes", case_themes)
    _themelist = ", ".join(c.replace("zs_is_", "") for c in _zt_is)
    print(f"registered case_themes ({len(case_themes)} rows, all sources; zero-shot themes: {_themelist})")
except FileNotFoundError:
    case_themes = None
    print("case_themes: themes_zeroshot.parquet not found (run theme_classify_zeroshot.py)")

# make the German corpora visible to the NL->SQL translator
SCHEMA = SCHEMA + (
    " Additional tables: swiss_meta(id TEXT, canton TEXT two-letter Swiss canton e.g. "
    "ZH=Zuerich, BE=Bern, AG=Aargau, BL=Basel-Land, BS=Basel-Stadt, GR=Graubuenden, "
    "SG=St.Gallen, CH=federal, court_type TEXT, year INT) = Swiss decisions; "
    "ris_meta(id TEXT, dokumenttyp TEXT[Rechtssatz|Text], year INT, senate TEXT) "
    "= Austrian OGH records."
    # same two readings of a country name the Bucket-1 router separates (route_sources):
    # the national corpus is a TABLE choice, the ECHR respondent is a WHERE clause, and the
    # translator picks the wrong one unless it is told which phrasing means which.
    " WHICH TABLE a country name means: 'in Austria', 'Austrian decisions', 'im "
    "oesterreichischen Recht', 'the OGH' = ris_meta (the Austrian corpus itself). "
    "'in Switzerland', 'Swiss decisions', 'das Bundesgericht' = swiss_meta. ONLY the "
    "respondent phrasing -- 'cases AGAINST Austria', 'X v. AUSTRIA', 'gegen Oesterreich' -- "
    "means the ECHR table with respondent_state='AUT' (Switzerland: 'CHE'). "
    "echr/echr_meta hold NO Austrian or Swiss domestic decisions, only judgments against "
    "those States.")
FEW_SHOT = FEW_SHOT + [
    ("Aus welchem Kanton stammen die meisten Schweizer Entscheidungen?",
     "SELECT canton, COUNT(*) AS n FROM swiss_meta GROUP BY canton ORDER BY n DESC LIMIT 5;"),
    ("How many Austrian decisions are in the corpus?",
     "SELECT COUNT(*) AS n FROM ris_meta;"),
    ("How many ECHR cases against Austria are there?",
     "SELECT COUNT(*) AS n FROM echr WHERE respondent_state = 'AUT';"),
]
SCHEMA = SCHEMA + (
    " keyword_hits(id TEXT, source TEXT[echr|swiss|ris], jurisdiction TEXT, keyword TEXT) = one "
    "row per (case, matched IMPORT keyword); joins to echr_meta/swiss_meta/ris_meta on id. Swiss "
    "& RIS keywords are German (Entfremdung, Kindeswohlgefaehrdung, Loyalitaetskonflikt, "
    "Kontaktverweigerung, Eltern-Kind-Entfremdung); ECHR keywords are English (parental "
    "alienation, contact rights, child abduction, custody of the child, visiting rights, ...). "
    "A matched keyword is NOT a classified theme -- see case_themes for classified topics.")
if case_themes is not None:
    SCHEMA = SCHEMA + (
        " case_themes(id TEXT, source TEXT[echr|swiss|ris], jurisdiction TEXT, primary_theme TEXT, "
        "theme_group TEXT, is_abduction_hague BOOL, is_adoption BOOL, is_care_removal BOOL, "
        "is_alienation BOOL, is_domestic_violence BOOL, is_custody_residence BOOL, "
        "is_contact_access BOOL, is_length_procedural BOOL) = ZERO-SHOT family-law THEMES for ALL "
        "sources (multilingual, approximate/high-recall -- NOT validated); joins to echr_meta/"
        "swiss_meta/ris_meta on id (+ source). Use this for theme counts across jurisdictions, e.g. "
        "contact_access per Swiss canton. echr.primary_theme/is_* are the ECHR-only regex themes.")
FEW_SHOT = FEW_SHOT + [
    ("Wie viele Schweizer Entscheidungen erwaehnen Kontaktverweigerung pro Kanton?",
     "SELECT s.canton, COUNT(DISTINCT k.id) AS n FROM keyword_hits k JOIN swiss_meta s "
     "ON k.id = s.id WHERE k.source = 'swiss' AND k.keyword = 'Kontaktverweigerung' "
     "GROUP BY s.canton ORDER BY n DESC;"),
]
if case_themes is not None:
    FEW_SHOT = FEW_SHOT + [
        ("How many contact_access cases are there per Swiss canton?",
         "SELECT s.canton, COUNT(*) AS n FROM case_themes t JOIN swiss_meta s ON t.id = s.id "
         "WHERE t.source = 'swiss' AND t.is_contact_access GROUP BY s.canton ORDER BY n DESC;"),
    ]
# ---- dynamic content fields: discover anything field_deploy.ipynb has shipped ----
# each deployed field = data/field_<name>_deployed.parquet + data/field_<name>_meta.json
# (definition, lexicon, threshold, validation). Registration + routing need NO code changes:
# define -> review -> deploy -> queryable here.
import glob as _glob
DEPLOYED_FIELDS = {}
for _mp in sorted(_glob.glob(str(DATA_DIR / "field_*_meta.json"))):
    _meta = _json.loads(open(_mp).read())
    _fname = _meta["field"]
    _pq = DATA_DIR / f"field_{_fname}_deployed.parquet"
    if not _pq.exists():
        continue
    _df = pd.read_parquet(_pq)
    con.register(f"field_{_fname}", _df)
    DEPLOYED_FIELDS[_fname] = _meta
    if _meta.get("kind") == "numeric":
        SCHEMA = SCHEMA + (
            f" Table field_{_fname}(id TEXT, {_fname} DOUBLE, conf_cal DOUBLE): calibrated "
            f"numeric content field in {_meta.get('unit', 'units')}; NULL means the case states "
            f"no value; trust a cell only where {_fname} IS NOT NULL AND conf_cal >= "
            f"{_meta['threshold']}.")
    else:
        SCHEMA = SCHEMA + (f" Table field_{_fname}(id TEXT, {_fname} BOOLEAN, conf_cal DOUBLE): "
                           f"calibrated content field, genuine = {_fname} AND conf_cal >= "
                           f"{_meta['threshold']}.")
if DEPLOYED_FIELDS:
    print("deployed content fields discovered:",
          {k: v["validation"] for k, v in DEPLOYED_FIELDS.items()})

print(f"registered swiss_meta ({len(swiss_meta)}) + ris_meta ({len(ris_meta)}) + "
      f"echr_meta ({len(echr_meta)}, year coverage "
      f"{echr_meta.year.notna().mean()*100:.0f}%, duration coverage "
      f"{echr_meta.duration_days.notna().mean()*100:.0f}%) + echr_kp ({len(echr_kp)}) "
      f"+ echr_hudoc ({len(echr_hudoc.columns)} raw HUDOC cols, scope-view only) | "
      f"schema + few-shot extended")

## 2b. Bucket 4 — diachronic wording change (keyness + grounded summary)
A fourth question class: **"what changed for topic T between X and Y?"** Top-k retrieval
structurally cannot answer it (no k passages carry a corpus-wide contrast), and free
generation would fabricate a narrative. The faithful path is the 5-step pipeline:

1. **topic grouping** — cases matched by a concept *cluster* (registry or question-derived);
2. **windowed keyness** — signed G2 over ±15 tokens around the anchors (on-topic,
   length-normalised — the denominator-artefact lesson from `diachronic_analysis.ipynb`);
3. **KWIC grounding** — real sentences with case ids for the top terms;
4. **constrained generation** — the local LLM sees *only* the evidence and may only
   describe/group it, never explain causes;
5. **faithfulness guard** — deterministic: every content word of the passage must appear in
   the keyness table or a quoted sentence, else the passage is *replaced* by the
   deterministic template. The guard, not the model, is the trust boundary — same
   philosophy as `EXPLAIN` validation for NL→SQL.

ECHR/English only (the dated corpus); German corpora are refused with the stated
rationale (undatable Rechtssätze, thin pre-2010 Swiss coverage). The output is a
**reading shortlist with a grounded description** — wording, not doctrine.

In [ ]:
# BUCKET 4 -- diachronic wording change: "what changed for <topic> between X and Y?"
# Pipeline: (1) topic grouping (concept cluster, not one keyword) -> (2) windowed keyness
# (signed G2, +/-15 tokens around anchors -- normalises corpus size by design) ->
# (3) KWIC grounding (real sentences from real cases) -> (4) constrained generation
# (local LLM given ONLY the evidence) -> (5) deterministic faithfulness guard: every
# content word of the passage must appear in the evidence (terms + quotes + scaffold);
# an ungrounded passage is REPLACED by the deterministic template, never shown.
# ECHR/English only -- the dated corpus. RIS Rechtssaetze are undatable and pre-2010
# Swiss coverage is thin, so a DE run is refused with that rationale, never attempted.
import json as _json
import re as _re
import math as _math
import urllib.request as _urlreq
from pathlib import Path
import pandas as pd

if "DATA_DIR" not in globals():
    DATA_DIR = Path("../data")

DIA_RANGE = (2000, 2025)          # corpus fact (extended 2026-07-07)
DIA_MIN_DOCS = 20                 # thin-window warning threshold (diachronic_analysis.ipynb)
DIA_NGRAM = (2, 3)                # keyness over 2-3 word PHRASES (collocations), not lone words
DIA_TOPICS = {                    # a topic is a CLUSTER of anchors; * = stem wildcard
    "parental alienation": ["alienat*", "estrang*", "parental alienation",
                            "loyalty conflict", "turning the child against",
                            "manipulat*", "undermin*"],
    "custody": ["custody", "sole custody", "joint custody", "residence order"],
    "contact": ["contact", "access", "visiting rights", "visitation"],
}

_DIA_TOK = _re.compile(r"[a-zA-ZäöüßÄÖÜ]+")   # letters only: anonymised "K._" must not yield "k_" tokens
_DIA_SENT = _re.compile(r"(?<=[.!?;])\s+")
_DIA_STOP = set(("the a an and or of to in on at for with without by from as is are was were be been "
                 "being this that these those it its their his her them they he she we you i me my our "
                 "your not no nor but if then than so such can could shall should may might must will "
                 "would do does did done has have had having also only very more most other others any "
                 "all each which who whom whose what when where how why there here under over between "
                 "into out about against during before after above below again further once because "
                 "until while both same some own too s t d ll m re ve see paragraph paragraphs inter alia ibid cf pp nos").split())
# content-free discourse scaffold the passage may use freely (never carries a claim)
_DIA_SCAFFOLD = set(("terms term wording vocabulary language period periods early late earlier later "
                     "rose rise rising faded fading fell became become becoming characteristic "
                     "distinctive around cases case shift shifted moved move moving toward towards "
                     "away appear appears appearing appeared passage passages meanwhile alongside "
                     "cluster clusters group groups keyness score scores frequent frequently often "
                     "corpus documents window windows versus overall within across first second half "
                     "notably particularly increasingly common usage use used phrases phrase words "
                     "word example instance texts text older newer register lists listed court began begin beginning contrast contrasts contrasting emerge emerges emerged emerging gradually like prominent prominently related relating instead whereas rather remained remain remains largely mostly mainly still similarly likewise marked markedly describe described describes describing present presented presence absence absent suggests suggesting indicates indicating reflects reflecting reflected together longer several fewer these those alongside gave give given gives made make makes took take taken turn turned came come less least gained gain gains lost lose losing "
                     "judgments decisions echr strasbourg").split())


def _dia_pat(anchor):
    a = anchor.strip().lower()
    if " " in a:
        return r"\b" + r"\s+".join(_re.escape(p) for p in a.split()) + r"\b"
    if a.endswith("*"):
        return r"\b" + _re.escape(a[:-1]) + r"\w*"
    return r"\b" + _re.escape(a) + r"\b"


def _dia_compile(anchors):
    combined = _re.compile("|".join("(?:%s)" % _dia_pat(a)
                                    for a in sorted(anchors, key=len, reverse=True)),
                           _re.IGNORECASE)
    singles = [_re.compile(r"^" + (_re.escape(a[:-1]) + r"\w*" if a.endswith("*")
                                   else _re.escape(a)) + r"$", _re.IGNORECASE)
               for a in anchors if " " not in a]
    return combined, singles


_DIA_CORPUS = None

def _dia_corpus():
    """Dated English ECHR docs, communicated cases dropped (Registry boilerplate
    would dominate keyness). Cached after first load."""
    global _DIA_CORPUS
    if _DIA_CORPUS is not None:
        return _DIA_CORPUS
    raw = (_echr_raw if "_echr_raw" in globals()
           else _json.loads((DATA_DIR / "echr_parental_alienation.json").read_text()))
    _dmy = _re.compile(r"(\d{1,2})/(\d{1,2})/(\d{4})")
    docs = []
    for r in raw:
        if (r.get("doctypebranch") or "") == "COMMUNICATEDCASES":
            continue
        if (r.get("conclusion") or "").strip().lower() == "communicated":
            continue
        year = None
        e = r.get("ecli") or ""
        if e.startswith("ECLI:CE:ECHR:") and e.split(":")[3][:4].isdigit():
            year = int(e.split(":")[3][:4])
        if year is None:
            for f in ("judgementdate", "decisiondate", "referencedate"):
                m = _dmy.match((r.get(f) or "").strip())
                if m:
                    year = int(m.group(3)); break
        text = r.get("full_text") or ""
        if year and text:
            docs.append({"id": r.get("itemid") or r.get("stable_id"),
                         "year": year, "text": text})
    _DIA_CORPUS = docs
    return docs


def _dia_topic(question):
    """Registry cluster whose name/anchor appears in the question; else anchors
    derived from the question's content words (stated in the output)."""
    q = question.lower()
    for name, anchors in DIA_TOPICS.items():
        probes = [name] + [a.rstrip("*") for a in anchors if len(a.rstrip("*")) > 5]
        if any(p in q for p in probes):
            return name, anchors, False
    drop = _DIA_STOP | _DIA_SCAFFOLD | {"changed", "change", "framed", "framing", "between",
                                        "wording", "vocabulary", "topic", "time"}
    words = [w for w in _DIA_TOK.findall(q) if w not in drop and not w.isdigit() and len(w) > 3]
    return (" ".join(words[:3]) or "unspecified topic"), [w + "*" for w in words[:5]], True


def _dia_windows(question):
    """Two comparison windows from the question's time expression. Resolution is
    always printed -- the user must see which years were actually compared."""
    lo, hi = DIA_RANGE
    b = _re.search(r"(?:before|until|up to)\s+((?:19|20)\d{2})", question, _re.I)
    a = _re.search(r"(?:after|since|from)\s+((?:19|20)\d{2})", question, _re.I)
    years = sorted({int(y) for y in _re.findall(r"\b((?:19|20)\d{2})\b", question)})
    if b and a:
        e1, l0 = int(b.group(1)), int(a.group(1))
        if e1 <= l0:
            return (lo, e1 - 1 if e1 == l0 else e1), (l0, hi), "explicit before/after"
    if len(years) >= 2:
        x, y = years[0], years[-1]
        m = (x + y) // 2
        return (x, m), (m + 1, y), f"interval {x}-{y} split at midpoint"
    if len(years) == 1:
        return (lo, years[0] - 1), (years[0], hi), f"split at {years[0]}"
    return (lo, 2012), (2013, hi), "default bins"


def _dia_keyness(early_docs, late_docs, combined, singles, window=15, min_freq=4):
    """Signed log-likelihood G2 over 2-3 word PHRASES within +/-window tokens of an
    anchor. Phrases are surface collocations (kept verbatim so KWIC + the guard still
    match); a phrase counts only when its whole span sits inside a window, its first
    and last words are content words (not stopwords/anchors), and it holds no digit."""
    def _is_anchor(tok):
        return any(p.match(tok) for p in singles)

    def _counts(docs):
        cnt = {}
        for d in docs:
            tk = [t.lower() for t in _DIA_TOK.findall(d["text"])]
            n = len(tk)
            inwin = set()
            for i, w in enumerate(tk):
                if _is_anchor(w):
                    inwin.update(range(max(0, i - window), min(n, i + window + 1)))
            if not inwin:
                continue
            for st in inwin:
                for size in DIA_NGRAM:
                    end = st + size
                    if end > n or (end - 1) not in inwin:
                        continue
                    seg = tk[st:end]
                    w0, wl = seg[0], seg[-1]
                    if (w0 in _DIA_STOP or wl in _DIA_STOP or _is_anchor(w0) or _is_anchor(wl)
                            or len(w0) <= 2 or len(wl) <= 2 or any(x.isdigit() for x in seg)):
                        continue
                    g = " ".join(seg)
                    cnt[g] = cnt.get(g, 0) + 1
        return cnt

    ec, lc = _counts(early_docs), _counts(late_docs)
    tot_l, tot_e = sum(lc.values()), sum(ec.values())
    rows = []
    if not tot_l or not tot_e:
        return rows
    for w in set(ec) | set(lc):
        a, b = lc.get(w, 0), ec.get(w, 0)
        if a + b < min_freq:
            continue
        e1 = tot_l * (a + b) / (tot_l + tot_e)
        e2 = tot_e * (a + b) / (tot_l + tot_e)
        ll = (a * _math.log(a / e1) if a else 0.0) + (b * _math.log(b / e2) if b else 0.0)
        g2 = 2 * ll
        if g2 < 3.84:                       # p < .05 cutoff -- below it, noise
            continue
        signed = g2 if (a / tot_l) >= (b / tot_e) else -g2
        rows.append({"term": w, "early": b, "late": a, "G2": round(g2, 1),
                     "signed_G2": round(signed, 1)})
    rows.sort(key=lambda r: r["signed_G2"], reverse=True)
    return rows


def _dia_quote(docs, term, combined, maxlen=220):
    """One REAL sentence containing the term, anchor-bearing sentences preferred."""
    tpat = _re.compile(r"\b" + _re.escape(term) + r"\b", _re.IGNORECASE)
    fallback = None
    for d in docs:
        for s in _DIA_SENT.split(d["text"]):
            s = _re.sub(r"\s+", " ", s).strip()
            if not (30 <= len(s) <= maxlen * 2) or not tpat.search(s):
                continue
            if combined.search(s):
                return s[:maxlen], d["id"]
            if fallback is None:
                fallback = (s[:maxlen], d["id"])
    return fallback


def _dia_llm(prompt):
    try:
        req = _urlreq.Request(
            "http://localhost:11434/api/generate",
            data=_json.dumps({"model": "llama3.2", "prompt": prompt, "stream": False,
                              "options": {"temperature": 0, "num_predict": 350}}).encode(),
            headers={"Content-Type": "application/json"})
        with _urlreq.urlopen(req, timeout=300) as r:  # CPU + cold model load
            return _json.loads(r.read()).get("response", "").strip() or None
    except Exception:
        return None


_DIA_SUF = ("ations", "ation", "ingly", "ences", "ements", "ement", "ments", "ment",
            "ance", "ence", "ent", "ities", "ity", "ing", "ied", "ies", "ed", "ly",
            "es", "s", "e")

def _dia_stem(w):
    """Tiny suffix-stripper so inflections of grounded words pass the guard
    (prominence/prominent, included/including) while foreign content nouns stay
    flagged. Min stem length 4 -- short words are never over-stripped."""
    for suf in _DIA_SUF:
        if w.endswith(suf) and len(w) - len(suf) >= 4:
            return w[:len(w) - len(suf)]
    return w


def _dia_guard(passage, allowed):
    """Deterministic net: content words not grounded in the evidence (stem-compared).
    The guard, not the model, is the trust boundary (same philosophy as EXPLAIN
    for NL->SQL)."""
    ok = {_dia_stem(w) for w in allowed} | {_dia_stem(w) for w in _DIA_SCAFFOLD}
    return sorted({w for w in (t.lower() for t in _DIA_TOK.findall(passage))
                   if len(w) >= 4 and not w.isdigit()
                   and w not in _DIA_STOP and _dia_stem(w) not in ok})


def _dia_template(topic, lab_e, lab_l, rose, faded, quotes):
    """Deterministic fallback passage -- built ONLY from the evidence, always guard-clean."""
    parts = [f"The wording around {topic} shifted between {lab_e} and {lab_l}."]
    if rose:
        parts.append("Terms that became more characteristic of the later period: "
                     + ", ".join(f"'{r['term']}' (G2 {r['G2']})" for r in rose) + ".")
        q = quotes.get(rose[0]["term"])
        if q:
            parts.append(f'For example: "...{q[0]}..." ({q[1]}).')
    if faded:
        parts.append("Terms that faded from use: "
                     + ", ".join(f"'{r['term']}' (G2 {r['G2']})" for r in faded) + ".")
        q = quotes.get(faded[0]["term"])
        if q:
            parts.append(f'Earlier usage: "...{q[0]}..." ({q[1]}).')
    return " ".join(parts)


def _h_diachronic(question, top_n=8):
    print("  BUCKET 4 (diachronic wording change -- keyness evidence + guarded summary)")
    if _re.search(r"austri\w*|swiss|schweiz\w*|österreich\w*|\bogh\b|german\w*|deutsch\w*",
                  question, _re.IGNORECASE):
        print("  REFUSED for the German corpora: RIS Rechtssaetze carry no reliable decision")
        print("  date and pre-2010 Swiss cantonal coverage is thin (stated corpus facts) --")
        print("  a temporal split would compare coverage artefacts, not wording.")
        print("  ECHR (English) is the dated corpus this question can be answered on.")
        return
    topic, anchors, derived = _dia_topic(question)
    combined, singles = _dia_compile(anchors)
    (e0, e1), (l0, l1), how = _dia_windows(question)
    docs = _dia_corpus()
    early = [d for d in docs if e0 <= d["year"] <= e1 and combined.search(d["text"])]
    late = [d for d in docs if l0 <= d["year"] <= l1 and combined.search(d["text"])]
    lab_e, lab_l = f"{e0}-{e1}", f"{l0}-{l1}"
    src_note = "derived from the question" if derived else "registry cluster"
    print(f"  topic: {topic} ({src_note}: {', '.join(anchors)})")
    print(f"  corpus: ECHR English, dated, communicated cases dropped ({len(docs)} docs)")
    print(f"  windows [{how}]: {lab_e} ({len(early)} mentioning docs) vs "
          f"{lab_l} ({len(late)} mentioning docs)")
    for lab, n in ((lab_e, len(early)), (lab_l, len(late))):
        if n < DIA_MIN_DOCS:
            print(f"  [!] WARNING: window {lab} has only {n} mentioning docs (<{DIA_MIN_DOCS}) "
                  f"-- shortlist, treat as anecdotal.")
    if not early or not late:
        print("  REFUSED: a window has no cases mentioning this topic -- no comparison "
              "is possible. Try a wider interval or a registry topic:",
              ", ".join(DIA_TOPICS))
        return
    rows = _dia_keyness(early, late, combined, singles)

    def _pick(cands, n):
        # drop a phrase when a stronger kept phrase nests it (share all words) --
        # so "county administrative court" doesn't also spend slots on its sub-phrases.
        kept = []
        for r in cands:
            toks = set(r["term"].split())
            if any(toks <= set(k["term"].split()) or set(k["term"].split()) <= toks for k in kept):
                continue
            kept.append(r)
            if len(kept) >= n:
                break
        return kept

    rose = _pick([r for r in rows if r["signed_G2"] > 0], top_n)
    faded = _pick(sorted((r for r in rows if r["signed_G2"] < 0), key=lambda r: r["signed_G2"]), top_n)
    if not rose and not faded:
        print("  NO statistically distinctive wording shift (all G2 < 3.84) -- the honest "
              "answer is 'no measurable change', not a narrative.")
        return
    print("  EVIDENCE -- windowed keyness of 2-3 word phrases (G2, +/-15 tokens around topic anchors):")
    ev = pd.DataFrame([{"direction": "rose", **{k: r[k] for k in ("term", "early", "late", "G2")}}
                       for r in rose]
                      + [{"direction": "faded", **{k: r[k] for k in ("term", "early", "late", "G2")}}
                         for r in faded])
    print("   " + ev.to_string(index=False).replace("\n", "\n   "))
    quotes = {}
    for r in rose[:3]:
        q = _dia_quote(late, r["term"], combined)
        if q:
            quotes[r["term"]] = q
    for r in faded[:3]:
        q = _dia_quote(early, r["term"], combined)
        if q:
            quotes[r["term"]] = q
    if quotes:
        print("  KWIC grounding (real sentences, case ids -- the passage may only quote these):")
        for t, (s, cid) in quotes.items():
            print(f"    '{t}': \"...{s}...\" ({cid})")
    # allowed vocabulary = evidence only; built BEFORE generation, checked after
    allowed = set()
    for r in rose + faded:
        allowed.update(t.lower() for t in _DIA_TOK.findall(r["term"]))
    for s, cid in quotes.values():
        allowed.update(t.lower() for t in _DIA_TOK.findall(s))
        allowed.add(cid.lower().replace("-", ""))
    allowed.update(t.lower() for t in _DIA_TOK.findall(topic))
    for a in anchors:
        allowed.update(t.lower() for t in _DIA_TOK.findall(a))
    template = _dia_template(topic, lab_e, lab_l, rose, faded, quotes)
    evidence = ("RISING TERMS (later window): "
                + "; ".join(f"{r['term']} (G2 {r['G2']})" for r in rose)
                + ". FADING TERMS (earlier window): "
                + "; ".join(f"{r['term']} (G2 {r['G2']})" for r in faded)
                + ". EXAMPLE PASSAGES: "
                + " | ".join(f"{t}: \"{s}\" ({cid})" for t, (s, cid) in quotes.items()))
    prompt = ("You summarise corpus-linguistics evidence about legal wording. Write ONE "
              f"paragraph (max 120 words) describing how the wording around '{topic}' "
              f"changed between {lab_e} and {lab_l}. STRICT RULES: mention only terms from "
              "the evidence, with their G2 scores in parentheses; you may quote the example "
              "passages verbatim; group related terms; do NOT name causes, events, laws or "
              "any term not listed; do NOT explain WHY anything changed.\n\nEVIDENCE: "
              + evidence + "\n\nPARAGRAPH:")
    passage, source = _dia_llm(prompt), "guarded local LLM (llama3.2, temp 0)"
    if passage is None:
        passage, source = template, "deterministic template (LLM unreachable)"
    else:
        bad = _dia_guard(passage, allowed)
        if bad:   # one retry with the violation named -- same pattern as EXPLAIN+retry
            retry = _dia_llm(prompt + "\n\nYour draft used words that are NOT in the "
                             "evidence: " + ", ".join(bad[:12]) + ". Rewrite the "
                             "paragraph without these words and without adding any "
                             "other new word.\n\nPARAGRAPH:")
            if retry:
                passage, bad = retry, _dia_guard(retry, allowed)
                source = "guarded local LLM (llama3.2, temp 0; passed on retry)"
            if bad:
                passage, source = template, (
                    "deterministic template (LLM draft REJECTED by guard "
                    f"{'twice' if retry else 'once'}, ungrounded: {', '.join(bad[:8])})")
    print(f"  PASSAGE [{source}]:")
    for line in _re.sub(r"\s+", " ", passage).strip().split(". "):
        print("    " + line.rstrip(".") + ".")
    leftover = _dia_guard(passage, allowed)
    print(f"  faithfulness guard: {'PASSED' if not leftover else 'FAILED ' + str(leftover)} "
          f"-- every content word grounded in the keyness table or a quoted sentence")
    print("  CAVEAT: a reading shortlist over the keyword-matched ECHR English corpus -- it "
          "describes WORDING, not doctrine, and never explains causes; windowed G2 "
          "normalises for corpus growth; counts are never litigation rates.")


# Bucket-4 trigger: a change-word AND a time signal. Both required, so Bucket-1
# framing questions ("How does the ECtHR frame...") and Bucket-2 year filters
# ("judgments after 2020") stay in their lanes.
_DIA_CHANGE = _re.compile(r"\b(chang\w*|shift\w*|evolv\w*|develop\w*|fram\w*|word\w*"
                          r"|vocabular\w*|language|differ\w*)\b", _re.IGNORECASE)
_DIA_TIME = _re.compile(r"\b(19|20)\d{2}\b|\bover time\b", _re.IGNORECASE)

def _diachronic_trigger(question):
    return bool(_DIA_CHANGE.search(question) and _DIA_TIME.search(question))


print(f"Bucket 4 ready -- topics: {', '.join(DIA_TOPICS)} (+ question-derived); "
      f"guard: evidence-only vocabulary, template fallback")

## 3. The dispatcher — content column, deny-list, then guarded NL→SQL
Order matters and each step is the *cheapest sufficient* mechanism:

**Bucket 1 first routes to a corpus.** A question that names a legal system is a question
about that legal system's law: "how is this scoped in Austria" is answered from the OGH, not
from ECHR judgments against Austria. `route_sources()` decides it before retrieval and
`_print_source_route()` prints which corpus answered and what the route dropped, because a
source restriction the reader cannot see is the same silent change of question as a hidden
metadata filter. The ECHR reading of a State stays reachable through the Court's own
phrasing — "cases **against** Austria", "X **v.** AUSTRIA".

1. **Validated content column** (`alienation_alleged`) — keyword-routed, because it must reach
   the calibrated-confidence machinery, and there is exactly one such column.
2. **Deny-list of known-unextracted concepts** (custody outcome, marital status, duration) —
   *semantic* knowledge about what is NOT in the data; no SQL validator can know it, and
   (measured) a 3B model given a refusal escape over-uses it on composable questions, so the
   model is never asked to refuse.
3. **Guarded NL→SQL** for everything else — the local coder model maps entities itself
   ("against Poland" → `respondent_state='POL'`, "Kanton Bern" → `swiss_meta`, canton `BE`),
   behind `EXPLAIN` validation with one retry. The generated SQL prints above its result —
   review the query, not the number.

Failure direction of every net is **refusal, never fabrication**.

In [ ]:
# ---------------------------------------------------------------------------
# Scope resolution -- read off the table schema, never from a hand-kept list
# ---------------------------------------------------------------------------
# Every filter added by hand cost a silent wrong answer first: a named respondent State was
# dropped, then a named year, each discovered only by asking a question whose answer was small
# enough to check. The columns of `echr` and `echr_meta` already state everything this corpus
# knows, so the resolver profiles them once at startup and derives the vocabulary it accepts.
# A metadata column added upstream becomes filterable and groupable with no edit here, and a
# condition the schema cannot honour is refused out loud instead of being dropped.

_DIM_CARD_MAX = 60      # more distinct values than this is not a category anyone names aloud
_DIM_MIN_LEN  = 4       # shorter values collide with ordinary words ('DEU', 'no', '8')
_DIM_ATOM_CAP = 400     # a ';'-list column with more atoms than this is free text, not a list


def _norm_q(text):
    """The one normalisation both sides of a match go through. Values were being compared raw
    against a normalised question, so HUDOC's "Art. 35-1" could never match the "art 35 1" the
    question had been reduced to -- the column simply looked unfilterable. Decimal points
    survive, because "conf 0.95" must not become "conf 0 95".
    """
    t = " " + _re.sub(r"[^a-z0-9_.]+", " ", str(text).lower()).strip() + " "
    return _re.sub(r"\s+", " ", _re.sub(r"(?<!\d)\.|\.(?!\d)", " ", t))


def _norm_v(text):
    return _norm_q(text).strip()


def _dim_key(col):
    """Column names that mean the same thing: `articles`/`article`, `separate_opinion`/
    `separateopinion`. Used to keep one table's column from shadowing another's."""
    k = col.lower().replace("_", "")
    return k[:-1] if k.endswith("s") else k


def _scope_view(base="echr", extra=("echr_meta", "echr_hudoc")):
    """One view carrying every ECHR metadata column, so "all metadata" is one join, not N.

    `echr` holds the extraction/theme layer, `echr_meta` the curated HUDOC layer (formation,
    duration, separate opinion) and `echr_hudoc` every remaining raw HUDOC field. Columns are taken from `extra` only where `base` lacks them, computed
    from the live schemas, so a column added to either table joins itself in. `year` stays the
    `echr` one deliberately: both carry it, and the deployed columns were validated on that one.
    """
    have = [r[0] for r in con.execute(f"DESCRIBE {base}").fetchall()]
    adds, joins = [], []
    for t in extra:
        try:
            cols = [r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()]
        except Exception:
            continue                                   # a corpus that is not loaded adds nothing
        # skip a column that restates one the base already has under a different spelling:
        # HUDOC `article` beside the extraction layer's `articles` would make the phrase
        # "article" ambiguous, and an ambiguous prefix names no column at all (_dim_prefixes)
        seen = {_dim_key(c) for c in have}
        new = [c for c in cols if c != "id" and c not in have and _dim_key(c) not in seen
               and not seen.add(_dim_key(c))]
        if not new:
            continue
        alias = f"x{len(joins)}"
        adds += [f'{alias}."{c}"' for c in new]
        joins.append(f"LEFT JOIN {t} {alias} ON b.id = {alias}.id")
        have += new
    con.execute(f"CREATE OR REPLACE VIEW echr_scope AS SELECT b.*"
                + "".join(", " + a for a in adds)
                + f" FROM {base} b " + " ".join(joins))
    return "echr_scope"


def _dim_values(table, col):
    return [r[0] for r in con.execute(
        f'SELECT DISTINCT "{col}" FROM {table} WHERE "{col}" IS NOT NULL').fetchall()]


def _dim_atoms(table, col):
    """The atoms of a ';'-delimited list column ('8;35' -> 8, 35), or [] if it is free text."""
    share = con.execute(f'''SELECT avg(CASE WHEN "{col}" LIKE '%;%' THEN 1.0 ELSE 0.0 END)
                            FROM {table} WHERE "{col}" IS NOT NULL''').fetchone()[0] or 0
    if share < 0.25:
        return []
    atoms = [r[0] for r in con.execute(
        f'''SELECT DISTINCT trim(u) FROM {table}, UNNEST(str_split("{col}", ';')) AS t(u)
            WHERE u IS NOT NULL AND trim(u) <> \'\'''').fetchall()]
    if len(atoms) > _DIM_ATOM_CAP or max((len(a) for a in atoms), default=99) > 60:
        return []
    return atoms


def _dim_is_date(table, col):
    """A column of ISO dates is not a set of categories: its values are numbers the date grammar
    already owns, and offering '1999-10-04' as a phrase to type helps nobody."""
    share = con.execute(f'''SELECT avg(CASE WHEN regexp_matches("{col}",
                              '^(\\d{{4}}-\\d{{2}}-\\d{{2}}|\\d{{2}}/\\d{{2}}/\\d{{4}})')
                            THEN 1.0 ELSE 0.0 END) FROM {table} WHERE "{col}" IS NOT NULL''').fetchone()[0]
    return (share or 0) >= 0.8


def _dim_registry(table):
    """Profile `table` into the dimensions a question may filter or group on.

    Kind is decided by the data, never by a name list, so nothing here knows what an ECHR
    column is called: bool -> a flag, cat -> few enough distinct values to be named aloud,
    num -> a number, list -> ';'-delimited multi-value. A high-cardinality string (title, url,
    evidence text, the JSON audit columns) offers nothing generic to match on, so it is left
    out and a question naming it is refused rather than silently ignored.
    """
    reg = {}
    for row in con.execute(f"DESCRIBE {table}").fetchall():
        col, typ = row[0], row[1].upper()
        nd = con.execute(f'SELECT count(DISTINCT "{col}") FROM {table}').fetchone()[0]
        if nd <= 1:
            continue        # one value for the whole corpus is not a dimension: filtering on
                            # `jurisdiction = 'Council of Europe'` selects everything and says
                            # nothing, and breaking down by it returns a single row
        if typ.startswith("BOOLEAN"):
            reg[col] = {"kind": "bool", "values": []}
            continue
        if any(k in typ for k in ("INT", "DOUBLE", "DECIMAL", "FLOAT", "REAL")):
            reg[col] = {"kind": "num",
                        "values": _dim_values(table, col) if nd <= _DIM_CARD_MAX else []}
        elif typ.startswith("VARCHAR"):
            if _dim_is_date(table, col):
                continue        # dates are the one dimension with a grammar; `year` owns them
            # a delimiter is stronger evidence than cardinality: HUDOC's `nonviolation` has 59
            # distinct values and every one of them is a ';'-list of article numbers, so reading
            # it as a category would offer "6-1;8" as a phrase to type
            if (atoms := _dim_atoms(table, col)):
                reg[col] = {"kind": "list", "values": atoms}
            elif nd <= _DIM_CARD_MAX:
                reg[col] = {"kind": "cat", "values": _dim_values(table, col)}
    return reg


def _dim_names(reg):
    """Phrase -> column for the column NAMES, including whatever _GROUP_SYNONYMS already
    teaches Bucket 2 ('country', 'state', 'theme'), so both buckets accept the same wording."""
    names = {}
    for col in reg:
        names[col.lower()] = names[col.replace("_", " ").lower()] = col
        if col.startswith(("is_", "has_")):
            bare = col.split("_", 1)[1]
            names.setdefault(bare.lower(), col)
            names.setdefault(bare.replace("_", " ").lower(), col)
    for phrase, cols in (globals().get("_GROUP_SYNONYMS") or {}).items():
        for c in cols:
            if c in reg:
                names.setdefault(phrase.lower(), c)
                break
    return names


def _dim_surfaces(reg):
    """Value phrase -> [(column, value)] for every value the schema says exists.

    Longest phrase wins at match time, so 'no violation' is never re-read as 'violation', and a
    phrase carried by two columns ('domestic violence' is both a `primary_theme` value and the
    `is_domestic_violence` flag) keeps both candidates -- the question decides between them.
    """
    surf = {}
    for col, d in reg.items():
        if d["kind"] == "cat":
            for v in d["values"]:
                for s in {_norm_v(v), _norm_v(str(v).replace("_", " "))}:
                    if len(s) >= _DIM_MIN_LEN:
                        surf.setdefault(s, []).append((col, v))
        elif d["kind"] == "bool":
            forms = {_norm_v(col), _norm_v(col.replace("_", " "))}
            if col.startswith(("is_", "has_")):               # `is_adoption` is said "adoption"
                bare = col.split("_", 1)[1]
                forms |= {_norm_v(bare), _norm_v(bare.replace("_", " "))}
            for s in forms:
                if len(s) >= _DIM_MIN_LEN:
                    surf.setdefault(s, []).append((col, True))
    return surf


def _dim_prefixes(reg):
    """Column -> the phrases that name it before a value ('article 8', 'importance 1').

    A shorthand is kept only where it is unambiguous: 'conf' names `outcome_conf`,
    `alienation_conf` and `alienation_conf_cal`, so it names none of them and the question
    must say which. Dropping ambiguity here is why two columns can never both fire on one
    phrase -- which they did, silently, before this was worked out.
    """
    cand = {}
    for col, d in reg.items():
        if d["kind"] not in ("num", "list", "cat"):
            continue
        spaced = col.replace("_", " ").lower()
        forms = {spaced} | set(spaced.split(" "))          # 'duration days', 'duration', 'days'
        for f in list(forms):
            if f.endswith("s"):
                forms.add(f[:-1])                          # 'article' for a column called articles
        cand[col] = {f for f in forms if len(f) >= 3}
    seen = {}
    for col, forms in cand.items():
        for f in forms:
            seen.setdefault(f, []).append(col)
    return {col: {f for f in forms if len(seen[f]) == 1} for col, forms in cand.items()}


def _sql_lit(v):
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


# grammar, not vocabulary: the words that may sit between a column's name and its value.
# "violation of 8-1" must not lose the 8-1 just because "of" came between them.
_DIM_CONNECTOR = r"(?:of|for|in|on|under|with|equal to|=)?"

_NUM_OPS = [("at least", ">="), ("no less than", ">="), ("no more than", "<="),
            ("at most", "<="), ("more than", ">"), ("greater than", ">"), ("over", ">"),
            ("above", ">"), ("less than", "<"), ("fewer than", "<"), ("under", "<"),
            ("below", "<")]


def _dim_pick(cands, q, names):
    """Which column a phrase carried by several of them means here. 'domestic violence' is a
    `primary_theme` value and an `is_domestic_violence` flag: naming the column in the question
    ('primary theme domestic violence') picks that column, otherwise the flag wins, because
    'cases involving X' asks whether X is present, not whether it is the single top theme.
    Whichever is taken is printed on the SCOPE line, so the reader sees the reading, not a guess.
    """
    for col, val in cands:
        for phrase, target in names.items():
            if target == col and len(phrase) >= 5 and f" {phrase} " in q:
                return col, val
    for col, val in cands:
        if _DIMS.get(col, {}).get("kind") == "bool":
            return col, val
    return cands[0]


def _scope_group(q):
    """The column a breakdown asks for: (column, matched_phrase, was_explicitly_asked).

    Resolved BEFORE any value matching, because the two compete for the same words -- 'per
    domestic violence' had its words eaten by the value scan and came back as no grouping at
    all. 'per X' and 'for each X' are a request and are refused when unresolvable; a bare 'by X'
    is not, because "decided by the Grand Chamber" is not asking for a breakdown.
    """
    for pat, explicit in ((r"\b(?:per|for each|broken down by)\s+([a-z0-9_][a-z0-9_ ]{2,40})", True),
                          (r"\bby\s+([a-z0-9_][a-z0-9_ ]{2,40})", False)):
        for m in _re.finditer(pat, q):
            words = m.group(1).split()
            for n in range(min(4, len(words)), 0, -1):
                key = " ".join(words[:n])
                if key in _DIM_NAMES:
                    return _DIM_NAMES[key], key, explicit
            if explicit:
                return None, m.group(1).strip(), True
    return None, None, False


def _field_phrases(field_terms):
    """The words of the field's own name, which are not metadata values -- but only where the
    schema does not need them. Masking token by token, `expert_opinion_ordered` erased the bare
    word 'opinion', and with it the `separate_opinion` flag, from every question about that
    field. So: multi-word phrases always, single words only when no dimension uses them.
    """
    toks = _re.findall(r"[a-z]{3,}", (field_terms or "").lower())
    protected = {w for s in _DIM_SURFACES for w in s.split()}
    protected |= {w for n in _DIM_NAMES for w in n.split()}
    protected |= {w for d in _DIMS.values() if d["kind"] == "list"
                  for a in d["values"] for w in str(a).lower().split()}
    grams = [" ".join(toks[i:i + n]) for n in range(len(toks), 1, -1)
             for i in range(len(toks) - n + 1)]
    return grams + [t for t in toks if t not in protected]


def _deployed_scope(question, field_terms=""):
    """What a question asks for beyond a corpus-wide count: (where_sql, group_col, note), or
    (None, None, reason) when it names a condition the schema cannot honour.

    Every dimension comes from `_DIMS`, profiled from `echr_scope` at startup, so this function
    contains no list of columns and needs no edit when the corpus gains one.
    """
    if not question:
        return "", None, ""
    cols, where, notes, used = set(_DIMS), "", [], set()

    # normalise, keeping decimal points: stripping them turned "conf 0.95" into "conf 0 95"
    # and filtered on a confidence of zero.
    q = _norm_q(question)
    for phrase in _field_phrases(field_terms):          # the field's own name is not a value
        q = _re.sub(rf"\b{_re.escape(phrase)}\b", " ", q)

    # 1. the breakdown, first, and its phrase is removed so it cannot also be read as a filter
    grp, gphrase, gexplicit = _scope_group(q)
    groupable = sorted(c for c in cols if _DIMS[c]["kind"] in ("bool", "cat", "num"))
    if gexplicit and (grp is None or grp not in cols):
        return None, None, (f"the question asks for a breakdown per {gphrase}, which "
                            f"`echr_scope` does not carry; it can break down by: "
                            f"{', '.join(groupable)}")
    if gphrase:
        q = q.replace(f" {gphrase} ", "  ")

    # 2. respondent State: a country NAME -> ISO code needs the learned map Bucket 1 uses, which
    #    no amount of schema profiling can replace.
    try:
        _, must = _sql_route(question)
    except Exception:
        must = []
    for col, val in must:
        if col not in cols:
            return None, None, f"the question names {col} = {val!r}, which `echr_scope` does not carry"
        if col in used:
            continue
        where += f" AND e.{col} = {_sql_lit(val)}"
        notes.append(f"{col} = {_sql_lit(val)}")
        used.add(col)

    # 3. list columns ('article 8', 'keyword parental alienation'): the column is named first and
    #    then one of its atoms, so an atom as common as "8" can never fire on its own.
    for col in sorted(c for c in cols if _DIMS[c]["kind"] == "list" and c not in used):
        atoms = {_norm_v(a): a for a in _DIMS[col]["values"] if _norm_v(a)}
        span = max((len(a.split()) for a in atoms), default=1)
        for p in sorted(_DIM_PREFIXES.get(col, ()), key=len, reverse=True):
            # a connector may sit between the column and its value ("violation OF 8-1"); without
            # this the value was dropped and only the bare word 'violation' survived, as an
            # `outcome` filter -- a different question, answered confidently
            m = _re.search(rf"\b{_re.escape(p)}s?\b\s*{_DIM_CONNECTOR}\s+"
                           r"([a-z0-9][a-z0-9 _]{0,80})", q)
            if not m:
                continue
            words = m.group(1).split()
            for n in range(min(span, len(words)), 0, -1):
                cand = " ".join(words[:n])
                if cand in atoms:
                    where += f" AND (';' || e.{col} || ';') LIKE '%;{atoms[cand]};%'"
                    notes.append(f"{col} contains '{atoms[cand]}'")
                    used.add(col)
                    # consume exactly what matched, connector included: rebuilding the phrase
                    # as "<name> <value>" missed the connector form, left the column's own name
                    # behind, and it was then matched a second time as a category value
                    q = q[:m.start()] + "  " + q[m.start(1) + len(cand):]
                    break
            if col in used:
                break

    # 3b. a category the question NAMES, which lets its short values through. The 4-character
    #     floor exists to stop a bare 'DEU' or '55' matching ordinary prose; naming the column
    #     removes that collision, and without this "applicability 55" resolved to nothing at all
    #     and answered over the whole corpus.
    for col in sorted(c for c in cols if _DIMS[c]["kind"] == "cat" and c not in used):
        vals = {_norm_v(v): v for v in _DIMS[col]["values"] if _norm_v(v)}
        span = max((len(a.split()) for a in vals), default=1)
        for p in sorted(_DIM_PREFIXES.get(col, ()), key=len, reverse=True):
            m = _re.search(rf"\b{_re.escape(p)}s?\b\s*{_DIM_CONNECTOR}\s+"
                           r"([a-z0-9][a-z0-9 _.]{0,80})", q)
            if not m:
                continue
            words = m.group(1).split()
            for n in range(min(span, len(words)), 0, -1):
                cand = " ".join(words[:n])
                if cand in vals:
                    where += f" AND e.{col} = {_sql_lit(vals[cand])}"
                    notes.append(f"{col} = {_sql_lit(vals[cand])}")
                    used.add(col)
                    q = q[:m.start()] + "  " + q[m.start(1) + len(cand):]
                    break
            if col in used:
                break

    # 4. numeric columns ('importance 1', 'duration over 2000 days'): a bare number is read as a
    #    filter only when the question also names the column it belongs to.
    ops = "|".join(_re.escape(o) for o, _ in sorted(_NUM_OPS, key=lambda x: -len(x[0])))
    for col in sorted(c for c in cols if _DIMS[c]["kind"] == "num" and c not in used):
        for p in sorted(_DIM_PREFIXES.get(col, ()), key=len, reverse=True):
            # the column is named before the number ("importance at least 2") or the number
            # carries it as a unit after ("over 2000 days"); either way a number alone is inert
            m = (_re.search(rf"\b{_re.escape(p)}\b\s*(?:of|is|was|level|value)?\s*({ops})?\s*"
                            r"(\d+(?:\.\d+)?)\b", q)
                 or _re.search(rf"({ops})?\s*(\d+(?:\.\d+)?)\s*\b{_re.escape(p)}s?\b", q))
            if not m:
                continue
            op = dict(_NUM_OPS).get(m.group(1), "=")
            where += f" AND e.{col} {op} {m.group(2)}"
            notes.append(f"{col} {op} {m.group(2)}")
            used.add(col)
            q = q.replace(m.group(0), "  ")
            break

    # 5. dates, last among the number-claiming steps. Ordering and ranges are a grammar, not a
    #    vocabulary, so they cannot be derived from distinct values the way every other dimension
    #    is -- but a bare four-digit number is only a year if no column has already claimed it.
    #    Read too early, "duration over 2000 days" filtered on `year = 2000` and said so with a
    #    straight face.
    if "year" in cols and "year" not in used:
        yr = None
        if (m := _re.search(r"between\s+((?:19|20)\d{2})\s+and\s+((?:19|20)\d{2})", q)):
            yr = (f"e.year BETWEEN {m.group(1)} AND {m.group(2)}", f"year in {m.group(1)}-{m.group(2)}")
        elif (m := _re.search(r"\b(?:since|from|in or after)\s+((?:19|20)\d{2})", q)):
            yr = (f"e.year >= {m.group(1)}", f"year >= {m.group(1)}")
        elif (m := _re.search(r"\bafter\s+((?:19|20)\d{2})", q)):
            yr = (f"e.year > {m.group(1)}", f"year > {m.group(1)}")
        elif (m := _re.search(r"\b(?:until|up to|through|by)\s+((?:19|20)\d{2})", q)):
            yr = (f"e.year <= {m.group(1)}", f"year <= {m.group(1)}")
        elif (m := _re.search(r"\bbefore\s+((?:19|20)\d{2})", q)):
            yr = (f"e.year < {m.group(1)}", f"year < {m.group(1)}")
        if yr:
            where += f" AND {yr[0]}"
            notes.append(yr[1])
            used.add("year")

    # 6. every categorical value and boolean flag the schema knows, longest phrase first so a
    #    match consumes its own words and 'no violation' is never re-read as 'violation'.
    for s in sorted(_DIM_SURFACES, key=len, reverse=True):
        span = f" {s} " if f" {s} " in q else None
        if span is None:
            # a value stored as one token is said as several: HUDOC writes the formation
            # `GRANDCHAMBER`, every person writes "Grand Chamber". Match the question's adjacent
            # words with the spaces closed up, which needs no list of how values are spelled.
            words = q.split()
            joined = {"".join(words[i:i + n]): " ".join(words[i:i + n])
                      for n in (2, 3) for i in range(len(words) - n + 1)}
            span = f" {joined[s]} " if s in joined else None
        if span is None:
            continue
        cands = [(c, v) for c, v in _DIM_SURFACES[s] if c not in used]
        if not cands:
            continue
        col, val = _dim_pick(cands, q, _DIM_NAMES)
        where += f" AND e.{col} = {_sql_lit(val)}"
        notes.append(f"{col} = {_sql_lit(val)}")
        used.add(col)
        q = q.replace(span, "  ")

    # 7. an unqualified four-digit number is a year only once nothing else has claimed it. The
    #    value scan runs first for the same reason the numeric scan does: `publishedby` reads
    #    "Reports of Judgments and Decisions 2013", and reading the year first filtered on both.
    if "year" in cols and "year" not in used and \
            (m := _re.search(r"(?<!\d)((?:19|20)\d{2})(?!\d)", q)):
        where += f" AND e.year = {m.group(1)}"
        notes.append(f"year = {m.group(1)}")

    return where, grp, ("filtered on " + " and ".join(notes)) if notes else ""


def scope_help(max_values=10):
    """Every dimension a deployed-field question can filter or group on, and the values it
    accepts. The vocabulary IS the data, so this is the authoritative answer to "what can I
    ask?" -- and it re-derives itself when the corpus gains a column or a value."""
    print(f"`echr_scope` -- {len(_DIMS)} filterable/groupable dimensions "
          f"(profiled at startup; nothing here is hand-listed)")
    for col, d in sorted(_DIMS.items(), key=lambda kv: (kv[1]["kind"], kv[0])):
        vals = d["values"][:max_values]
        shown = ", ".join(str(v) for v in vals) if vals else ""
        more = f" (+{len(d['values']) - len(vals)} more)" if len(d["values"]) > len(vals) else ""
        if d["kind"] == "bool":
            shown = "true / false"
        elif d["kind"] == "num" and not vals:
            shown = "any number, with over / under / at least / at most"
        print(f"  {col:22s} {d['kind']:5s} {shown}{more}")
    print("  values are matched as they are stored: the theme `abduction_hague` answers to that "
          "phrase, not to the bare word 'abduction'.")


_SCOPE_VIEW   = _scope_view()
_DIMS         = _dim_registry(_SCOPE_VIEW)
_DIM_NAMES    = _dim_names(_DIMS)
_DIM_SURFACES = _dim_surfaces(_DIMS)
_DIM_PREFIXES = _dim_prefixes(_DIMS)
print(f"scope resolver ready -- {_SCOPE_VIEW}: {len(_DIMS)} dimensions "
      f"({sum(1 for d in _DIMS.values() if d['kind'] == 'cat')} categorical, "
      f"{sum(1 for d in _DIMS.values() if d['kind'] == 'bool')} flags, "
      f"{sum(1 for d in _DIMS.values() if d['kind'] == 'num')} numeric, "
      f"{sum(1 for d in _DIMS.values() if d['kind'] == 'list')} list), "
      f"{len(_DIM_SURFACES)} value phrases | scope_help() lists them")


def _h_alienation(q):
    """BUCKET 3 (extracted): the one validated content column, threshold-aware."""
    res, audit = alienation_at(min_confidence=MIN_CONFIDENCE, mode="filter")
    print("  BUCKET 3 (content aggregate -- extracted + calibrated)")
    print(f"  ANSWER: {audit['passed']} cases with a confident alienation allegation "
          f"(calibrated conf >= {audit['min_confidence']}); "
          f"{audit['abstained_low_conf']} further predicted-positive cells ABSTAINED "
          f"(low confidence, surfaced not dropped); basis={audit['confidence_basis']}")
    print(res[["id", "title", "respondent_state", "conf_eff"]].head(5).to_string(index=False))
    print("  CAVEAT: extractor validated at F1 0.51-0.59 on 120 gold labels; ECHR Article-8 "
          "English cases only -- a calibrated estimate, not a hard fact.")


REFUSAL = ("BUCKET 3 (content aggregate -- NOT extracted): no validated per-case column "
           "carries this answer (e.g. custody outcome, marital status). "
           "Answering from retrieval would fabricate a statistic; the faithful options are "
           "building + validating an extractor for that field (see echr_extraction.ipynb) "
           "or this refusal.")

# net 1 -- known-unextracted content concepts (semantic knowledge SQL validation cannot have):
_UNEXTRACTED = _re.compile(
    r"(receiv\w*|erhielt|erhält|awarded|granted|zugesprochen)\s+(the\s+)?(sole\s+)?"
    r"(custody|sorgerecht|obhut)"
    r"|(custody|sorgerecht|obhut)\s+(was\s+)?(receiv|award|grant|zugesprochen)"
    r"|\bmarried\b|\bverheiratet\b",
    _re.IGNORECASE)   # duration removed 2026-07-07: introductiondate arrived,
                      # proceedings duration moved from refusal to Bucket 2

MAX_RESULT_ROWS = 40  # display cap for Bucket-2 results; truncation is announced, never silent


_ISO_NAMES = {"NOR": "norway", "RUS": "russia", "POL": "poland", "ROU": "romania",
              "UKR": "ukraine", "HUN": "hungary", "DEU": "german", "ITA": "ital",
              "HRV": "croatia", "BGR": "bulgaria", "TUR": "turk", "AUT": "austria",
              "CHE": "switzerland", "FRA": "france", "GRC": "greece", "LTU": "lithuania",
              "MLT": "malta", "MDA": "moldova", "SVN": "slovenia", "SWE": "sweden",
              "GBR": "united kingdom", "NLD": "netherlands", "ESP": "spain", "PRT": "portugal",
              "CZE": "czech", "SVK": "slovak", "FIN": "finland", "IRL": "ireland",
              "BEL": "belgium", "SRB": "serbia", "LVA": "latvia", "EST": "estonia"}


def _unrequested_filters(sql, question):
    """Transparency net: every literal the SQL filters on should trace back to the question.
    Measured failure -- 'per primary_theme against Romania' produced
    `AND primary_theme != 'abduction_hague'`, a restriction nobody asked for, returning a
    plausible-looking number. Heuristic and non-blocking: it warns, it does not refuse."""
    q = (question or "").lower()
    unasked = []
    for col, lit in _re.findall(r"(\w+)\s*(?:=|!=|<>)\s*'([^']+)'", sql):
        low = lit.lower()
        if low in q or low.replace("_", " ") in q:
            continue
        if _ISO_NAMES.get(lit.upper(), "\0") in q or lit.lower() in q:
            continue
        unasked.append(f"{col} {'!=' if '!=' in sql or '<>' in sql else '='} '{lit}'")
    if unasked:
        print(f"  CHECK THE SQL: it filters on {', '.join(unasked)} -- no such restriction "
              f"appears in your question. Verify this is what you meant before using the number.")


def _coverage_note(sql):
    """Transparency net: for every table.column the generated SQL touches, report
    rows with no value there. A question phrased against an incomplete column
    silently drops those rows from filters and groupings -- the user must see that."""
    tables = set(_re.findall(r"\b(?:FROM|JOIN)\s+([A-Za-z_]\w*)", sql, _re.IGNORECASE))
    sql_words = {w.lower() for w in _re.findall(r"[A-Za-z_]\w*", sql)}
    for t in sorted(tables):
        try:
            cols = [r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()]
        except Exception:
            continue
        for c in cols:
            if c.lower() not in sql_words or c.lower() == "id":
                continue
            n, filled = con.execute(f'SELECT COUNT(*), COUNT("{c}") FROM {t}').fetchone()
            if filled < n:
                print(f"  COVERAGE: {t}.{c} has a value in {filled}/{n} rows -- the other "
                      f"{n - filled} rows are invisible to any filter or grouping on it.")


# tables per corpus, so the hint can name what the translator may actually SELECT from
# The hint names the PRIMARY table only. Naming case_themes/keyword_hits here as well was
# measured to make the translation WORSE, not better: those are the only two tables carrying a
# `jurisdiction` column, and mentioning them led the model to filter ris_meta by a column
# ris_meta does not have -- "How many Austrian decisions are in the corpus?" (a verbatim
# FEW_SHOT entry, correct without the hint) failed as
#   Binder Error: Referenced column "jurisdiction" not found in FROM clause!
# Measured across the 12-question matrix: 10/12 valid with the old hint, 12/12 without it and
# 12/12 with the primary-table-only hint below. The theme/keyword tables stay described in
# SCHEMA, where they are reachable when a question actually needs them.
_SQL_ROUTE_TABLES = {
    "echr":  "the ECHR corpus (tables echr, echr_meta, echr_kp)",
    "ris":   "the Austrian OGH corpus (table ris_meta)",
    "swiss": "the Swiss corpus (table swiss_meta)",
}


def _sql_route(question):
    """Hand Bucket 2 the same routing decision Bucket 1 makes -> (prose hint, required filters).
    The translator cannot derive it: 'Austrian decisions' and 'cases against Austria' name the
    same State and two different corpora. The respondent code goes back as a REQUIRED FILTER,
    not as advice -- measured, the prose hint alone was ignored and 'how many cases against
    Switzerland per year' still came back as a whole-corpus per-year count."""
    rt = route_sources(question)
    if not rt["sources"]:
        return "", []
    parts = ["This question is about " +
             " and ".join(_SQL_ROUTE_TABLES[s] for s in rt["sources"]) +
             f" -- {'; '.join(rt['basis'])}. Do not answer it from another corpus's table. "
             "That table IS the corpus: it has no jurisdiction, source or country column, so "
             "do not add a filter for the country -- only for conditions the question states."]
    must = [("respondent_state", c) for c in sorted(rt["countries"])]
    if must:
        parts.append("It names ECHR respondent State " + "/".join(c for _, c in must) +
                     ": the SQL must filter echr.respondent_state (or echr_meta.respondent) "
                     "on that code.")
    return " ".join(parts), must


def _h_nl2sql(q):
    """BUCKET 2 (metadata aggregate) via guarded NL->SQL. Three deterministic nets:
    the source route (below, computed by rule and handed to the translator), the deny-list
    (above, checked inside nl2sql too) and EXPLAIN validation + retry.
    Refusal is never delegated to the model (measured lazy-escape effect)."""
    hint, must = _sql_route(q)
    if hint:
        print("  SOURCE ROUTE (handed to the translator):", hint)
    sql = nl2sql(q, hint=hint, must_filter=must)
    if sql is None:
        print("  BUCKET 2 (metadata aggregate) -- UNAVAILABLE, not refused:")
        print(f"  the NL->SQL translator is unreachable on backend "
              f"'{globals().get('GEN_BACKEND', 'ollama')}': "
              f"{gen_ready()[1] if 'gen_ready' in globals() else 'ollama not running'}.")
        print("  The question IS answerable from metadata; see the canned queries in echr_query.ipynb.")
        return
    if sql.startswith("--"):
        if sql.startswith("-- GROUPING_MISMATCH"):       # valid SQL, wrong question
            print("  BUCKET 2 (metadata aggregate) -- REFUSED, translator answered a "
                  "different question:")
            print("   ", sql)
            print("  An ungrouped total in place of a breakdown is a silent substitution: the "
                  "number looks right and answers something else. Ask for the breakdown "
                  "explicitly (\"count per primary_theme\"), or use a canned query.")
            return
        if sql.startswith("-- CANNOT_ANSWER"):            # genuine content deny-list
            print(f"  NL->SQL declined: {sql}")
            print(" ", REFUSAL); return
        # invalid / failed translation -> SCHEMA mismatch, NOT a content refusal
        print("  BUCKET 2 (metadata aggregate) -- could not translate to valid SQL over the schema:")
        print("   ", sql)
        print("  Likely the question mixes fields that never co-exist in one table.")
        print("  Queryable: echr_meta / echr (ECHR -- respondent, year, importance, genre, outcome, themes); "
              "swiss_meta (canton, court_type, year); ris_meta (dokumenttyp, senate, year).")
        print("  NOTE: cross-source THEMES are in case_themes (zero-shot, approximate) -- e.g. contact_access "
              "per Swiss canton: JOIN case_themes to swiss_meta on id WHERE source='swiss' AND is_contact_access. "
              "Raw matched import KEYWORDS are in keyword_hits. echr.is_*/primary_theme are ECHR-only regex themes.")
        return
    print("  BUCKET 2 (metadata aggregate) -- generated SQL (REVIEW THIS, it is the trust boundary):")
    print("   ", sql)
    try:
        out = run_sql(sql)
        print(out.head(MAX_RESULT_ROWS).to_string(index=False))
        if len(out) > MAX_RESULT_ROWS:
            print(f"  ... {len(out) - MAX_RESULT_ROWS} more rows not shown "
                  f"(display cap {MAX_RESULT_ROWS}; add LIMIT/ORDER BY to the question to narrow)")
    except Exception as e:
        print("  execution failed:", e); print(" ", REFUSAL); return
    _unrequested_filters(sql, q)
    _coverage_note(sql)
    print("  CAVEAT: counts describe the keyword-matched corpora, never litigation rates; "
          "citable numbers come from the reviewed canned queries in echr_query.ipynb.")


def _match_deployed_fields(question):
    """Every deployed content field a question names, in the order it names them."""
    # Token matching, not substring: a term's content words must all appear in the question,
    # in any order. Contiguous matching missed "how many ECHR cases have father as an
    # applicant?" for `applicant_is_father`, whose terms are "applicant is father" and
    # "fathers" — both present in meaning, neither present as a phrase.
    _STOP = {"is", "the", "a", "an", "of", "or", "in", "to", "was", "were", "be", "who"}

    def _toks(text):
        return {w[:-1] if len(w) > 3 and w.endswith("s") else w          # crude singularise
                for w in _re.findall(r"[a-z]+", text.lower())} - _STOP

    qt = _toks(question)
    ql = (question or "").lower()
    hits = {}
    for fname, meta in DEPLOYED_FIELDS.items():
        terms = [fname.replace("_", " ")] + list(meta.get("question_terms", []))
        terms += [t for t in meta.get("lexicon", []) if len(t) > 5]
        for t in terms:
            tt = _toks(t)
            if tt and tt <= qt:
                at = min((m.start() for w in _re.findall(r"[a-z]{3,}", t.lower())
                          if (m := _re.search(rf"\b{_re.escape(w)}\b", ql))), default=len(ql))
                hits[fname] = min(hits.get(fname, len(ql)), at)
    return sorted(hits, key=lambda f: (hits[f], f))   # by earliest mention, name breaking ties


def _match_deployed_field(question):
    """The first field a question names, or None -- the single-field entry point."""
    got = _match_deployed_fields(question)
    return got[0] if got else None


def _h_deployed_numeric(fname, meta, thr, question=""):
    """BUCKET 3 for a numeric deployed field: the distribution, not a count of Trues."""
    unit = meta.get("unit", "")
    print(f"  BUCKET 3 (content aggregate -- extracted + calibrated, numeric field `{fname}`)")
    # a numeric field is scoped by the same resolver as a boolean one: this handler used to
    # query `field_<name>` alone, so every filter a question named was dropped by construction.
    where, grp, note = _deployed_scope(question, field_terms=fname)
    if where is None:
        print(f"  CANNOT ANSWER AS ASKED: {note}. Re-ask without that condition, call "
              f"scope_help() for the dimensions and values this corpus does carry, or use "
              f"SQL directly.")
        return
    if note:
        print(f"  SCOPE: {note} (joined to `echr_scope` on case id)")
    if grp:                                # a resolved breakdown that is ignored is a wrong answer
        rows = run_sql(f"""SELECT e.{grp} AS {grp},
                                  COUNT(*) FILTER (f.{fname} IS NOT NULL AND f.conf_cal >= {thr}) AS confident,
                                  median(f.{fname}) FILTER (f.conf_cal >= {thr}) AS median_value,
                                  COUNT(*) AS cases
                           FROM field_{fname} f JOIN echr_scope e ON f.id = e.id
                           WHERE TRUE{where}
                           GROUP BY 1 ORDER BY confident DESC NULLS LAST LIMIT 25""")
        print(f"  ANSWER: median {fname} per {grp} over cells at calibrated conf >= {thr} "
              f"({unit}); cells below the bar are excluded from the median, not imputed")
        print(rows.to_string(index=False))
        v = meta["validation"]
        print(f"  CAVEAT: LLM extractor validated on {v['n_labels']} reviewed labels "
              f"(accuracy {v.get('accuracy')}, MAE {v.get('mae')} {unit}, flip rate "
              f"{v['flip_rate']}); domain of validity: {meta['corpus'].split(' ')[0]}.")
        return
    res = run_sql(f"""SELECT COUNT(*) FILTER (f.{fname} IS NOT NULL AND f.conf_cal >= {thr}) AS confident,
                             COUNT(*) FILTER (f.{fname} IS NOT NULL AND f.conf_cal <  {thr}) AS abstained,
                             COUNT(*) AS corpus,
                             median(f.{fname}) FILTER (f.conf_cal >= {thr}) AS med,
                             min(f.{fname}) FILTER (f.conf_cal >= {thr}) AS lo,
                             max(f.{fname}) FILTER (f.conf_cal >= {thr}) AS hi
                      FROM field_{fname} f JOIN echr_scope e ON f.id = e.id
                      WHERE TRUE{where}""")
    conf_n, abst_n = int(res.confident[0]), int(res.abstained[0])
    v = meta["validation"]
    if conf_n:
        print(f"  ANSWER: {conf_n} of {int(res.corpus[0])} cases carry a confident {fname} "
              f"(calibrated conf >= {thr}) -- median {res.med[0]:.0f} {unit}, range "
              f"{res.lo[0]:.0f}-{res.hi[0]:.0f}; {abst_n} further extracted values ABSTAINED "
              f"(low confidence, surfaced not dropped); the rest state no value")
        top = run_sql(f"""SELECT f.id, e.title, f.{fname} AS value, f.conf_cal
                          FROM field_{fname} f JOIN echr_scope e ON f.id = e.id
                          WHERE f.{fname} IS NOT NULL AND f.conf_cal >= {thr}{where}
                          ORDER BY f.conf_cal DESC LIMIT 5""")
        print(top.to_string(index=False))
    else:
        print(f"  ANSWER: none. {abst_n} extracted values all fall below the trust bar "
              f"(conf_cal >= {thr}), so the field abstains rather than reporting them")
    print(f"  CAVEAT: LLM extractor validated on {v['n_labels']} reviewed labels "
          f"(accuracy {v.get('accuracy')}, MAE {v.get('mae')} {unit}, flip rate "
          f"{v['flip_rate']}); domain of validity: {meta['corpus'].split(' ')[0]}.")


def _h_deployed_field(fname, question=""):
    """BUCKET 3 (extracted + calibrated) for a factory-deployed field — generic handler."""
    meta = DEPLOYED_FIELDS[fname]
    thr = meta["threshold"]
    if meta.get("kind") == "numeric":
        return _h_deployed_numeric(fname, meta, thr, question)
    print(f"  BUCKET 3 (content aggregate -- extracted + calibrated, field `{fname}`)")
    where, grp, note = _deployed_scope(question, field_terms=fname)
    if where is None:                      # a named condition this column cannot honour
        print(f"  CANNOT ANSWER AS ASKED: {note}. Re-ask without that condition, call "
              f"scope_help() for the dimensions and values this corpus does carry, or use "
              f"SQL directly.")
        return
    if note:
        print(f"  SCOPE: {note} (joined to `echr_scope` on case id)")
    if grp:
        rows = run_sql(f"""SELECT e.{grp} AS {grp},
                                  COUNT(*) FILTER (f.{fname} AND f.conf_cal >= {thr}) AS confident,
                                  COUNT(*) FILTER (f.{fname} AND f.conf_cal <  {thr}) AS abstained,
                                  COUNT(*) AS cases
                           FROM field_{fname} f JOIN echr_scope e ON f.id = e.id
                           WHERE TRUE{where}
                           GROUP BY 1 ORDER BY confident DESC NULLS LAST LIMIT 25""")
        print(f"  ANSWER: confident {fname.replace('_', ' ')} per {grp} "
              f"(calibrated conf >= {thr}); abstained cells surfaced, not dropped")
        print(rows.to_string(index=False))
        v = meta["validation"]
        print(f"  CAVEAT: LLM extractor validated on {v['n_labels']} reviewed labels "
              f"(F1 {v.get('llm_f1')}, flip rate {v['flip_rate']}); "
              f"domain of validity: {meta['corpus'].split(' ')[0]}.")
        return
    res = run_sql(f"""SELECT COUNT(*) FILTER (f.{fname} AND f.conf_cal >= {thr}) AS confident,
                             COUNT(*) FILTER (f.{fname} AND f.conf_cal <  {thr}) AS abstained,
                             COUNT(*) AS corpus
                      FROM field_{fname} f JOIN echr_scope e ON f.id = e.id
                      WHERE TRUE{where}""")
    conf_n, abst_n = int(res.confident[0]), int(res.abstained[0])
    v = meta["validation"]
    print(f"  ANSWER: {conf_n} cases with a confident {fname.replace('_', ' ')} "
          f"(calibrated conf >= {thr}); {abst_n} further predicted-positive cells ABSTAINED "
          f"(low confidence, surfaced not dropped)"
          + (f"; scope = {int(res.corpus[0])} cases" if note else ""))
    top = run_sql(f"""SELECT f.id, e.title, f.conf_cal FROM field_{fname} f
                      JOIN echr_scope e ON f.id = e.id
                      WHERE f.{fname} AND f.conf_cal >= {thr}{where}
                      ORDER BY f.conf_cal DESC LIMIT 5""")
    print(top.to_string(index=False))
    print(f"  CAVEAT: LLM extractor validated on {v['n_labels']} reviewed labels "
          f"(F1 {v['llm_f1']}, flip rate {v['flip_rate']}); "
          f"domain of validity: {meta['corpus'].split(' ')[0]}.")


def _h_deployed_fields(fnames, question=""):
    """BUCKET 3 over SEVERAL deployed fields: a question naming two columns is asking for their
    conjunction, and answering about one of them is the silent-drop fault of section 5.5 one
    level up -- "both an expert opinion ordered AND the child heard" came back as a plain
    `child_heard` count, with nothing in the output to say the other half had been discarded.

    Each field keeps its OWN calibrated threshold, because each was calibrated separately; a
    case counts only where every field is true and every cell clears its own bar.
    """
    metas = {f: DEPLOYED_FIELDS[f] for f in fnames}
    nums = [f for f in fnames if metas[f].get("kind") == "numeric"]
    if len(nums) > 1:
        print(f"  CANNOT ANSWER AS ASKED: the question names {len(nums)} numeric fields "
              f"({', '.join(nums)}), which have no single distribution between them; ask about "
              f"one at a time, optionally filtered by the other.")
        return
    print(f"  BUCKET 3 (content aggregate -- {len(fnames)} extracted fields combined: "
          f"{' AND '.join('`' + f + '`' for f in fnames)})")
    where, grp, note = _deployed_scope(question, field_terms=" ".join(fnames))
    if where is None:
        print(f"  CANNOT ANSWER AS ASKED: {note}. Re-ask without that condition, call "
              f"scope_help() for the dimensions this corpus carries, or use SQL directly.")
        return
    if note:
        print(f"  SCOPE: {note} (joined to `echr_scope` on case id)")

    al = {f: f"f{i}" for i, f in enumerate(fnames)}
    frm = (f"FROM field_{fnames[0]} {al[fnames[0]]} "
           + " ".join(f"JOIN field_{f} {al[f]} ON {al[fnames[0]]}.id = {al[f]}.id"
                      for f in fnames[1:])
           + f" JOIN echr_scope e ON {al[fnames[0]]}.id = e.id")
    true_of = {f: (f"{al[f]}.{f} IS NOT NULL" if f in nums else f"{al[f]}.{f}") for f in fnames}
    conf_of = {f: f"{al[f]}.conf_cal >= {metas[f]['threshold']}" for f in fnames}
    all_true = " AND ".join(true_of[f] for f in fnames)
    all_conf = " AND ".join(conf_of[f] for f in fnames)

    res = run_sql(f"""SELECT COUNT(*) FILTER ({all_true} AND {all_conf}) AS confident,
                             COUNT(*) FILTER ({all_true} AND NOT ({all_conf})) AS abstained,
                             COUNT(*) AS scope
                      {frm} WHERE TRUE{where}""")
    conf_n, abst_n, scope_n = (int(res.confident[0]), int(res.abstained[0]), int(res.scope[0]))
    print(f"  ANSWER: {conf_n} cases where ALL of "
          f"{', '.join(f.replace('_', ' ') for f in fnames)} hold confidently (each at its own "
          f"calibrated threshold); {abst_n} further cases where all are predicted but at least "
          f"one cell ABSTAINS (surfaced, not dropped); scope = {scope_n} cases")

    # the components, over the same scope: a small conjunction is otherwise unreadable
    parts = run_sql(f"""SELECT {', '.join(f'COUNT(*) FILTER ({true_of[f]} AND {conf_of[f]}) AS "{f}"'
                                          for f in fnames)}
                        {frm} WHERE TRUE{where}""")
    print("  components (same scope, each field alone): "
          + " · ".join(f"{f} {int(parts[f][0])}" for f in fnames))

    if nums:
        n = nums[0]
        dist = run_sql(f"""SELECT median({al[n]}.{n}) AS med, min({al[n]}.{n}) AS lo,
                                  max({al[n]}.{n}) AS hi
                           {frm} WHERE TRUE{where} AND {all_true} AND {all_conf}""")
        if conf_n:
            print(f"  {n}: median {dist.med[0]:.0f} {metas[n].get('unit', '')}, range "
                  f"{dist.lo[0]:.0f}-{dist.hi[0]:.0f} over those {conf_n} cases")
    if conf_n:
        top = run_sql(f"""SELECT {al[fnames[0]]}.id, e.title,
                                 {', '.join(f'{al[f]}.conf_cal AS "conf_{f}"' for f in fnames)}
                          {frm} WHERE TRUE{where} AND {all_true} AND {all_conf}
                          ORDER BY {' + '.join(f'{al[f]}.conf_cal' for f in fnames)} DESC LIMIT 5""")
        print(top.to_string(index=False))
    print("  CAVEAT: each field carries its own validation -- "
          + " · ".join(f"{f} F1 {metas[f]['validation'].get('llm_f1')} "
                       f"(flip {metas[f]['validation']['flip_rate']})" for f in fnames)
          + ". A conjunction is LESS reliable than either column: the errors do not cancel, "
            "they accumulate, so treat this count as the weakest of the two claims, not the "
            "strongest. Domain of validity: "
          + metas[fnames[0]]["corpus"].split(" ")[0] + ".")


SHOW_HITS = 10   # console cap; the full retrieved set is always in the provenance JSON


def _print_grounding(r):
    """Print the three counts separately. 'How many cases is this grounded in' has one honest
    answer -- cases_cited -- and it is usually far smaller than the number retrieved."""
    g, st = r.get("grounding") or {}, r.get("retrieval_stats") or {}
    if not g:
        return
    rng = st.get("score_range")
    print(f"  GROUNDING: retrieved {g['chunks_retrieved']} chunks / {g['cases_retrieved']} cases"
          f"{f' (cos {rng[0]}-{rng[1]})' if rng else ''}"
          f" | in prompt {g['chunks_in_prompt']} chunks / {g['cases_in_prompt']} cases"
          f" ({g.get('chunks_in_prompt_full', 0)} full + {g.get('chunks_in_prompt_extract', 0)}"
          f" extract, packing={g.get('packing')})"
          f" | CITED in the answer {g['chunks_cited']} chunks / {g['cases_cited']} cases")
    if st.get("mode") == "per_source":
        print(f"  RECALL AUDIT: one search per corpus, each with its own cut "
              f"(cross-lingual scores sit lower by construction, so a shared cut would "
              f"delete the very passages a comparison needs)")
        return
    print(f"  RECALL AUDIT: {st.get('pool_fetched', 0)} candidates scored -> "
          f"-{st.get('dropped_genre', 0)} communicated, "
          + (f"-{st['dropped_out_of_source']} in another corpus, "
             if st.get("dropped_out_of_source") else "")
          + (f"-{st['dropped_out_of_scope']} out of scope, "
             if st.get("dropped_out_of_scope") else "")
          + (f"-{st['dropped_mmr_pool']} beyond the MMR pool cap, "
             if st.get("dropped_mmr_pool") else "")
          + f"-{st.get('dropped_below_cut', 0)} below the cut "
          f"(cos {st.get('cut')} = {st.get('cut_basis')}, top {st.get('top_cos')}), "
          f"-{st.get('dropped_case_cap', 0)} over {MAX_CHUNKS_PER_CASE}/case"
          + ("" if st.get("pool_exhausted") else
             f"; pool capped at fetch_k={st.get('pool_ceiling')}, further matches exist below"))
    if g.get("jurisdictions_retrieved"):
        st_src = (st.get("per_source") or {})
        if st_src:
            print("  PER-SOURCE RETRIEVAL: " + " | ".join(
                f"{k}: {v.get('kept', 0)} chunks (top cos {v.get('top_cos')})"
                for k, v in st_src.items()))
        elif len(g["jurisdictions_retrieved"]) == 1:
            # after a source route this is the route doing its job, not an accident of ranking;
            # saying "single-jurisdiction evidence base" for both would blur the difference
            routed = (r.get("route") or {}).get("sources")
            print(f"  NOTE: every retrieved passage is {g['jurisdictions_retrieved'][0]} -- "
                  + ("as routed (see SOURCE ROUTE above)." if routed else
                     "a single-jurisdiction evidence base for this answer, and nothing in the "
                     "question asked for that."))
    if g.get("chunks_in_prompt") and not g.get("chunks_cited"):
        # measured degeneration signature: placeholder citations ("[a][b][c]"), enumerated
        # source dumps, or generic prose with nothing pointing at a passage
        print("  WARNING: the answer cites nothing that resolves to a source in the prompt -- "
              "treat it as ungrounded prose, not as an answer.")
    for m in g.get("jurisdiction_mismatches", []):
        print(f"  WARNING: answer attributes {m['chunk_id']} to {m['claimed']}, but it is "
              f"{m['actual']} -- misattribution across jurisdictions")
    if g.get("citations_real_but_unseen"):
        print(f"  WARNING: the answer cites real corpus ids the generator never saw: "
              f"{g['citations_real_but_unseen']}")
    if g.get("citations_invented"):
        print(f"  WARNING: the answer cites ids that exist nowhere in the corpus "
              f"(invented): {g['citations_invented']}")


def _print_source_route(r):
    """Which corpus answered, and why. Without this line a single-jurisdiction question looks
    like a whole-corpus answer that happened to come out of one court."""
    rt = (r.get("route") or {})
    st = r.get("retrieval_stats") or {}
    if not rt.get("sources"):
        print("  SOURCE ROUTE: none -- the question names no legal system, so all three "
              "corpora were searched together (ECHR + Austrian OGH + Swiss).")
        return
    labels = ", ".join(SOURCE_LABELS.get(s, s) for s in rt["sources"])
    if r.get("comparative_sources"):
        print(f"  SOURCE ROUTE: {labels} -- {len(rt['sources'])} legal systems named, so "
              f"retrieval ran once per corpus ({'; '.join(rt.get('basis') or [])}).")
        return
    print(f"  SOURCE ROUTE: {labels} only ({'; '.join(rt.get('basis') or [])}) -- "
          f"{st.get('dropped_out_of_source', 0)} candidate chunks from the other corpora were "
          f"dropped BEFORE the similarity cut; {st.get('in_source', 0)} remained. "
          f"Ask with sources=set() to search all three instead.")


def _print_scope(r):
    """A metadata restriction the reader cannot see is a silent change of question."""
    sc = (r.get("scope") or {})
    st = r.get("retrieval_stats") or {}
    if not sc.get("countries"):
        return
    if r.get("comparative_sources"):
        # the question names jurisdictions, so per-source routing already governs the split;
        # reporting a respondent-State filter that never ran would be a phantom restriction
        print(f"  SCOPE: not applied -- the question names "
              f"{len(r['comparative_sources'])} jurisdictions, so retrieval ran once per "
              f"corpus instead of filtering by respondent State.")
        return
    print(f"  SCOPE: respondent State = {'/'.join(sc['countries'])} "
          f"(matched '{', '.join(sc.get('matched') or [])}' in the question) -- "
          f"{st.get('dropped_out_of_scope', 0)} candidate chunks against other States were "
          f"dropped BEFORE the similarity cut; {st.get('in_scope', 0)} remained in scope. "
          f"Ask with scope=set() to search the whole corpus instead.")


def _print_cases_relied_on(r):
    """The id -> case legend, built from chunk METADATA (never from the model's text).

    A synthesis groups the evidence by proposition, so the case names no longer appear in
    reading order -- the reader still needs one place that says which judgment each cited id
    is, and what its section and respondent State were."""
    cited = (r.get("grounding") or {}).get("cited_chunk_ids") or []
    if not cited:
        return
    by_id = {h["chunk_id"]: h for h in r.get("used", [])}
    per_case = {}
    for cid in cited:
        h = by_id.get(cid)
        if h:
            per_case.setdefault((h["source"], h["id"]), {"h": h, "ids": []})["ids"].append(cid)
    print(f"  CASES RELIED ON ({len(per_case)}) -- id -> case, from metadata:")
    for e in per_case.values():
        h = e["h"]
        print(f"    {h['title'][:52]:52s} | {h.get('date') or 'n.d.':10s} | "
              f"{(h.get('country') or '-'):7s} | {h.get('section', '') or '-':10s} | "
              f"{', '.join(e['ids'])}")
        print(f"      {h['url']}")


def _h_direct_sql(sql):
    """BUCKET 2, manual. Type SQL and it runs verbatim — no model, no translation, no
    reinterpretation. Same guardrail as generated SQL (SELECT/WITH only, EXPLAIN-checked),
    because the danger is a typo against the schema, not the author's intent."""
    print("  BUCKET 2 (metadata aggregate) -- YOUR SQL, run verbatim:")
    print("   ", sql)
    err = validate_sql(sql)
    if err:
        print("  rejected:", err)
        print("  Queryable: echr (themes + extracted fields) · echr_meta (year, formation, "
              "duration) · echr_kp · case_themes (all 3 sources) · swiss_meta · ris_meta")
        return
    try:
        out = run_sql(sql)
    except Exception as e:
        print("  execution failed:", e); return
    print(out.head(MAX_RESULT_ROWS).to_string(index=False))
    if len(out) > MAX_RESULT_ROWS:
        print(f"  ... {len(out) - MAX_RESULT_ROWS} more rows not shown (display cap)")
    _coverage_note(sql)
    print("  CAVEAT: counts describe the keyword-matched corpora, never litigation rates.")


def ask_anything(question, k=TOP_K):
    print("=" * 88)
    print("Q:", question)
    if _re.match(r"\s*(SELECT|WITH)\b", question or "", _re.IGNORECASE):
        print("ROUTE: starts with SELECT/WITH -> your SQL, executed as written")
        _h_direct_sql(question.strip())
        return
    if _diachronic_trigger(question):
        print("ROUTE: Bucket 4 (change-over-time trigger) -> keyness evidence + "
              "guarded summary (never free generation)")
        _h_diachronic(question)
        return
    trig = aggregate_trigger(question)
    if not trig:
        print(f"ROUTE: Bucket 1 (no aggregate trigger) -> retrieval + grounded generation "
              f"(style={GEN_ANSWER_STYLE})")
        r = answer(question, k=k)
        _print_source_route(r)
        _print_scope(r)
        if r["abstained"]:
            print(f"  abstained: {r['abstain_type']} -- {r['answer'][:200]}")
        elif r["answer"]:
            print("  ANSWER:", r["answer"].replace("\n", " "))
        else:
            print("  (generation OFF -- LLM unreachable; retrieval-only mode, hits below)")
        _print_grounding(r)
        _print_cases_relied_on(r)
        cited = set((r.get("grounding") or {}).get("cited_chunk_ids", []))
        in_prompt = {h["chunk_id"] for h in r.get("used", [])}
        for n, h in enumerate(r["hits"][:SHOW_HITS], 1):
            # cited > in the prompt but unused > retrieved only
            flag = ("cited " if h["chunk_id"] in cited else
                    "prompt" if h["chunk_id"] in in_prompt else "recall")
            print(f"  [{n:2d}] {flag} cos={h['score']:.3f} | {h['jurisdiction']:9s} | "
                  f"{(h.get('country') or '-'):7s} | {h['title'][:48]}")
        if len(r["hits"]) > SHOW_HITS:
            print(f"  ... {len(r['hits']) - SHOW_HITS} further retrieved chunks not printed")
        print("  PROVENANCE:", save_provenance(r))
        return
    print(f"ROUTE: aggregate trigger '{trig}' -> query layer (never generation)")
    if _re.search(r"alienat|entfremd", question, _re.IGNORECASE):
        _h_alienation(question)                      # validated content column first
        return
    flds = _match_deployed_fields(question)          # factory-deployed fields, auto-routed
    if len(flds) > 1:
        _h_deployed_fields(flds, question)           # a question naming two of them means both
    elif flds:
        _h_deployed_field(flds[0], question)
    elif _UNEXTRACTED.search(question):
        print(" ", REFUSAL)                          # known-unextracted concept
    else:
        _h_nl2sql(question)                          # semantic metadata path


print("ask_anything() ready -- dispatch: diachronic -> content column -> deny-list -> guarded NL->SQL")
print(f"  Bucket 1: source routing={SOURCE_ROUTING}, respondent scoping={SCOPE_BY_RESPONDENT}, "
      f"answer style={GEN_ANSWER_STYLE}")

## 4. One entry point, nine questions, five distinct behaviours
Two Bucket-1 (answered with citations, one DE one EN), one aggregate in each metadata
dimension, the extracted content aggregate with its abstention audit, one question the
system **correctly refuses**, and one Bucket-4 diachronic wording-change question
(keyness evidence + guarded summary).

In [ ]:
DEMO = [
    "When can custody be transferred to the other parent because of alienating behaviour?",
    "Unter welchen Voraussetzungen kann einem Elternteil die Obhut entzogen werden?",
    "How many cases against Poland are in the corpus?",              # NOT in few-shot
    "Wie viele Schweizer Entscheidungen stammen aus dem Kanton Bern?",  # NOT in few-shot
    "What share of merits judgments after 2020 found a violation?",  # composed condition
    "How many cases involve an allegation of parental alienation?",
    "How long do proceedings take on average from application to judgment?",  # refused until 2026-07-07
    "In what proportion of cases did the mother receive custody?",
    "How was parental alienation framed before 2015 and after 2015?",   # BUCKET 4 diachronic
]
for q in DEMO:
    ask_anything(q)

In [10]:
ask_anything("How many contact_access cases are there aganist Romania?")

Q: How many contact_access cases are there aganist Romania?
ROUTE: aggregate trigger 'How many' -> query layer (never generation)
  BUCKET 2 (metadata aggregate) -- generated SQL (REVIEW THIS, it is the trust boundary):
    SELECT COUNT(*) AS cases FROM echr WHERE respondent_state = 'ROU' AND primary_theme = 'contact_access';
 cases
     5
  CAVEAT: counts describe the keyword-matched corpora, never litigation rates; citable numbers come from the reviewed canned queries in echr_query.ipynb.
